In [2]:
import pandas as pd
from pathlib import Path
import csv
import time
import requests
from typing import List, Dict, Optional

In [ ]:
## Set up file paths
## project root is two levels above notebooks/raw_data_processing
PROJECT_ROOT = Path.cwd().parent.parent
#SHAPEFILE = PROJECT_ROOT / 'data' / 'raw' / 'fsa_boundary' / 'gfsa000a11a_e.shp'
SAVE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'average_heating_days_cleaned.csv'
RADON_DATA_AVAILABLE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'radon_concentration_cleaned.csv'
FSA_BOUNDARY_PATH = PROJECT_ROOT / 'data' / 'processed' / 'fsa_centroids.csv'

Function to make API call to openmeteo historical weather data

In [ ]:

def fetch_with_retry(url, params, max_retries=5):
    for _ in range(max_retries):
        response = requests.get(url, params=params)
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 429:
            wait_time = int(response.headers.get("Retry-After", 20))
            print(f"Rate limited. Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
        else:
            print(f"Error {response.status_code}: {response.text}")
            break
    return None


def fetch_historical_weather_multiple(
    latitudes: List[float],
    longitudes: List[float],
    start_date: str,
    end_date: str,
    target_variable: str,
    location_names: Optional[List[str]] = None,
) -> Dict[str, pd.DataFrame]:
    """
    Fetch historical weather data for multiple lat/lon pairs, including wind direction.

    Args:
        latitudes: List of latitudes.
        longitudes: List of longitudes.
        start_date: Start date in "YYYY-MM-DD" format.
        end_date: End date in "YYYY-MM-DD" format.
        location_names: Optional names for each location (default: "loc_0", "loc_1", ...).

    Returns:
        Dictionary of DataFrames (key: location name, value: weather data).
    """
    if len(latitudes) != len(longitudes):
        raise ValueError("Latitudes and longitudes must have the same length.")

    elif len(location_names) != len(latitudes):
        raise ValueError("Location names must match latitudes/longitudes length.")

    csv_path = SAVE_PATH
    file_exists = csv_path.exists()

    base_url = "https://archive-api.open-meteo.com/v1/archive"
    failed_locations = []
    counter = 1

    for lat, lon, name in zip(latitudes, longitudes, location_names):
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": start_date,
            "end_date": end_date,
            "daily": target_variable,
        }
        
        response = fetch_with_retry(base_url, params=params,max_retries=5)
        if response is None:         
            print(f"Failed to fetch data for {name} at ({lat}, {lon}). Skipping.")
            failed_locations.append((name, lat, lon))
            if len(failed_locations) >= 4:
                print("Too many failed locations. Stopping further requests.")
                return failed_locations
            continue
        print(f"{counter}. Successfully fetched data for {name} at ({lat}, {lon}).")
        df = pd.DataFrame(response['daily'], columns=['time', target_variable])

        
        heating_days_threshold = 18.0
        df['heating_day'] = df[target_variable] < heating_days_threshold
        average_for_location = df['heating_day'].mean() * 365

        # Write to CSV immediately after each location
        try:
            with open(csv_path, 'a', newline='') as f:
                writer = csv.writer(f)
                if not file_exists:
                    writer.writerow(['FSA', 'average_heating_days'])
                    file_exists = True
                try:
                    writer.writerow([name, average_for_location])
                except Exception as e:
                    print(f"Error writing to CSV for {name}: {e}")
        except Exception as e:
            print(f"Error opening CSV file for {name}: {e}")

        counter += 1
    return failed_locations

Make API call and save average number of heating days per year (days of the year with maximum temperature less than 18 $^{\circ}$ C)

In [ ]:
data_available_df = pd.read_csv(RADON_DATA_AVAILABLE_PATH)
coordinates_df = pd.read_csv(FSA_BOUNDARY_PATH)
locations_df = coordinates_df[coordinates_df['FSA'].isin(data_available_df['FSA'])]
failed_locations_all = []
start_date = "2002-01-01"
end_date = "2011-12-31"
target_variable = "temperature_2m_max"
chunk_size = 20     ## small chunk size to avoid rate limits
counter = 0
# for i in range(0, len(locations_df), chunk_size):
#     locations = locations_df['FSA'].tolist()[i:i+chunk_size]
#     latitude = locations_df['latitude'].tolist()[i:i+chunk_size]
#     longitude = locations_df['longitude'].tolist()[i:i+chunk_size]

#     failed_locations = fetch_historical_weather_multiple(latitude, longitude,
#                     start_date, end_date,target_variable =target_variable,
#                     location_names=locations)
#     failed_locations_all.extend(failed_locations)
#     print(f"Chunk {i//chunk_size + 1} completed. Failed locations so far: {len(failed_locations_all)}."
#           f"Sleeping to avoid rate limits...")
#     counter += 1
#     if counter % 3 == 0 and counter % 12 !=0: ## wait longer after every 3rd chunk
#         time.sleep(200)  
#     elif counter % 12 == 0:  ## even longer after every 12th chunk
#         time.sleep(600)
#     else:   
#         time.sleep(90) 

Manually fixing failed locations

In [5]:
locations = locations_df['FSA'].tolist()[992:1014]
latitude = locations_df['latitude'].tolist()[992:1014]
longitude = locations_df['longitude'].tolist()[992:1014]

failed_locations = fetch_historical_weather_multiple(latitude, longitude,
                start_date, end_date,target_variable =target_variable,
                location_names=locations)
failed_locations_all.extend(failed_locations)

1. Successfully fetched data for V9J at (49.72844281697943, -125.23978277561557).
2. Successfully fetched data for V9K at (49.35497568793841, -124.56252518205106).
3. Successfully fetched data for V9L at (48.77757555411864, -123.73717250362924).
4. Successfully fetched data for V9M at (49.70364730960831, -124.84283167178252).
5. Successfully fetched data for V9N at (49.65591097114184, -124.97190490459631).
6. Successfully fetched data for V9P at (49.24405489317608, -124.1953842582684).
7. Successfully fetched data for V9R at (49.1475356652335, -123.92409961440342).
8. Successfully fetched data for V9S at (49.19068485333346, -123.95600466488564).
9. Successfully fetched data for V9T at (49.21502670020869, -124.0148281579192).
10. Successfully fetched data for V9V at (49.293231231890566, -123.93342687147734).
11. Successfully fetched data for V9W at (50.07637858661779, -125.58517785945008).
12. Successfully fetched data for V9X at (49.10926402641167, -124.25402991951826).
13. Successfull